# Semantic Sink and Chain-Level Drift Analysis

This notebook focuses on the first sink question: do low-frequency abstract words drift toward high-frequency abstract attractors, e.g. `Solipsism -> Thought`?

The basic unit is a chain row: `model_name + pipeline + original_word + instance_id + step`. The notebook keeps chain identity and step identity, then aggregates only after chain-level summaries are computed.

Options implemented here:

1. Direct named sink test: `Solipsism -> Thought` hit rate, first-hit step, final-hit rate.
2. Exact vocabulary category flow: low-frequency abstract source words whose guesses exactly become high-frequency abstract vocabulary words.
3. Embedding basin flow: guesses mapped to the nearest high-frequency abstract vocabulary word using the existing cached embeddings.
4. Qualitative traces: step-by-step chains and guess transitions for writing about how drift happens.

Aggregation note: for category-level reports, compute source-word rates first, then average across words, matching the aggregation style used in `run_model_comparison_analysis.py`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'datasets').exists():
    raise FileNotFoundError('Could not find the project root containing datasets/.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_analysis.deeper_evaluation.semantic_sink_helpers import (
    CATEGORY_ORDER,
    MODEL_ORDER,
    load_frequency_lookup,
    chain_metric_deltas,
    aggregate_chain_deltas_word_first,
    add_guess_frequency_features,
    add_description_text_features,
    add_description_similarity_to_step0,
    add_word_guess_similarity,
    basin_rank_by_step,
    chain_summary_by_model,
    exact_sink_rank_by_step,
    guess_transitions,
    load_chain_rows,
    load_target_vocabulary,
    nearest_category_basin_map,
    normalize_word,
    qualitative_trace,
    step_profile,
    summarize_chain_hits,
    top_guesses_by_step,
    word_first_source_sink_rates,
    word_step_sink_rates,
)

UNIFIED_CSV = PROJECT_ROOT / 'datasets' / 'unified_semantic_drift_results.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'semantic_sinks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE = (
    PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'all_guess_similarity_description_similarity'
    / 'cache' / 'embeddings_sentence_transformers_all_minilm_l6_v2.sqlite'
)

SELECTED_MODELS = MODEL_ORDER
SELECTED_PIPELINES = ['pipeline_a', 'pipeline_b']
SOURCE_CATEGORY = 'Low-Freq-Abstract'
SINK_CATEGORY = 'High-Freq-Abstract'
SOURCE_WORD = 'Ennui'
SINK_WORD = 'Idea'
FINAL_STEP = 10
CHUNK_SIZE = 250_000
progress = tqdm if tqdm else None

UNIFIED_CSV.exists(), EMBEDDING_CACHE.exists(), OUTPUT_DIR


## Vocabulary Map

Exact sink flow only classifies guesses that match one of the 400 target words. Other guesses are handled later with the embedding-basin method.

In [ ]:
vocab = load_target_vocabulary(UNIFIED_CSV, chunk_size=CHUNK_SIZE, progress=progress)
word_to_category = dict(zip(vocab['word_norm'], vocab['category']))
word_to_original = dict(zip(vocab['word_norm'], vocab['original_word']))
high_freq_abstract_vocab = vocab[vocab['category'].eq(SINK_CATEGORY)]['word_norm'].sort_values().tolist()
low_freq_abstract_vocab = vocab[vocab['category'].eq(SOURCE_CATEGORY)]['word_norm'].sort_values().tolist()

vocab.to_csv(OUTPUT_DIR / 'target_vocabulary_with_categories.csv', index=False)
print(vocab['category'].value_counts().sort_index())
source_in_vocab = normalize_word(SOURCE_WORD) in word_to_category
sink_in_vocab = normalize_word(SINK_WORD) in word_to_category
print('Source word in vocab:', source_in_vocab, '|', SOURCE_WORD)
print('Sink word in vocab:', sink_in_vocab, '|', SINK_WORD)

if not source_in_vocab:
    source_prefix = normalize_word(SOURCE_WORD)[:3]
    source_suggestions = vocab[vocab['word_norm'].str.startswith(source_prefix, na=False)][['category', 'original_word']].head(20)
    print('Requested SOURCE_WORD is not a source word in the 400-word dataset. Matching-prefix suggestions:')
    display(source_suggestions)

if not sink_in_vocab:
    # Direct named-sink tests can still use a non-vocabulary sink if it appears as a guess,
    # but exact category-flow analysis can only classify sinks that are in the 400-word vocabulary.
    sink_prefix = normalize_word(SINK_WORD)[:3]
    sink_suggestions = vocab[vocab['word_norm'].str.startswith(sink_prefix, na=False)][['category', 'original_word']].head(20)
    print('Requested SINK_WORD is not in the 400-word vocabulary. Direct matching may still work if models guess it, but exact category/basin tables need vocabulary words.')
    display(sink_suggestions)

vocab.head()


# Option 1: Direct Named Sink Test

This directly answers: for chains starting from `Solipsism`, how often do they hit `Thought`, at what step, and do they end there?

In [ ]:
source_norm = normalize_word(SOURCE_WORD)
sink_norm = normalize_word(SINK_WORD)

case_rows = load_chain_rows(
    UNIFIED_CSV,
    word_to_category,
    source_words=[SOURCE_WORD],
    models=SELECTED_MODELS,
    pipelines=SELECTED_PIPELINES,
    include_descriptions=True,
    chunk_size=CHUNK_SIZE,
    progress=progress,
)
if case_rows.empty:
    available_lfa = vocab[vocab['category'].eq(SOURCE_CATEGORY)][['category', 'original_word']].sort_values('original_word')
    raise ValueError(
        f'No chain rows found for SOURCE_WORD={SOURCE_WORD!r}. '
        f'This usually means the word is not one of the 400 target words. '
        f'Examples from {SOURCE_CATEGORY}: {available_lfa["original_word"].head(20).tolist()}'
    )

case_rows['is_named_sink_guess'] = case_rows['guess_norm'].eq(sink_norm)
case_rows['is_sink_category_exact'] = case_rows['guess_category_exact'].eq(SINK_CATEGORY)
case_rows, missing_case_embeddings = add_word_guess_similarity(case_rows, EMBEDDING_CACHE)

case_rows.to_csv(OUTPUT_DIR / f'chain_rows_{source_norm}.csv', index=False)
print(case_rows.shape)
print('Missing embeddings:', len(missing_case_embeddings))
case_rows.head()


In [ ]:
named_sink_chain_summary = summarize_chain_hits(case_rows, sink_word_norm=sink_norm)
named_sink_by_model = chain_summary_by_model(named_sink_chain_summary, sink_label=f'{source_norm}_to_{sink_norm}')
case_named_step_profile = step_profile(case_rows, 'is_named_sink_guess', similarity_col='word_to_guess_similarity')
case_top_guesses = top_guesses_by_step(case_rows, top_n=10)

named_sink_chain_summary.to_csv(OUTPUT_DIR / f'named_sink_chain_summary_{source_norm}_to_{sink_norm}.csv', index=False)
named_sink_by_model.to_csv(OUTPUT_DIR / f'named_sink_by_model_{source_norm}_to_{sink_norm}.csv', index=False)
case_named_step_profile.to_csv(OUTPUT_DIR / f'named_sink_step_profile_{source_norm}_to_{sink_norm}.csv', index=False)
case_top_guesses.to_csv(OUTPUT_DIR / f'top_guesses_by_step_{source_norm}.csv', index=False)

named_sink_by_model

In [ ]:
case_named_step_profile

# Option 2: Exact Category Sink Flow

This asks the broader version: across all low-frequency abstract source words, how often do chains enter exact high-frequency abstract vocabulary guesses?

The chain summary gives first/final hits per chain. The model/category report first computes each source word's sink rate, then averages across words.

In [ ]:
lfa_rows = load_chain_rows(
    UNIFIED_CSV,
    word_to_category,
    source_categories=[SOURCE_CATEGORY],
    models=SELECTED_MODELS,
    pipelines=SELECTED_PIPELINES,
    include_descriptions=False,
    chunk_size=CHUNK_SIZE,
    progress=progress,
)
lfa_rows['is_hfa_exact_sink'] = lfa_rows['guess_category_exact'].eq(SINK_CATEGORY)
lfa_rows['is_known_vocab_guess'] = lfa_rows['guess_category_exact'].notna()

lfa_rows.to_csv(OUTPUT_DIR / 'low_freq_abstract_chain_rows_no_descriptions.csv', index=False)
print(lfa_rows.shape)
lfa_rows.head()

In [ ]:
category_sink_chain_summary = summarize_chain_hits(lfa_rows, sink_category=SINK_CATEGORY)
source_word_sink_rates, model_pipeline_hfa_flow = word_first_source_sink_rates(category_sink_chain_summary)
word_step_hfa_rates, category_step_hfa_flow = word_step_sink_rates(lfa_rows, 'is_hfa_exact_sink')

category_sink_chain_summary.to_csv(OUTPUT_DIR / 'lfa_to_hfa_exact_category_chain_summary.csv', index=False)
source_word_sink_rates.to_csv(OUTPUT_DIR / 'lfa_source_word_hfa_exact_sink_rates.csv', index=False)
model_pipeline_hfa_flow.to_csv(OUTPUT_DIR / 'lfa_to_hfa_exact_flow_by_model_pipeline.csv', index=False)
word_step_hfa_rates.to_csv(OUTPUT_DIR / 'lfa_word_step_hfa_exact_rates.csv', index=False)
category_step_hfa_flow.to_csv(OUTPUT_DIR / 'lfa_category_step_hfa_exact_flow.csv', index=False)

model_pipeline_hfa_flow

In [ ]:
category_step_hfa_flow

In [ ]:
source_step_sink_rates, hfa_sink_rank_by_step_top, hfa_sink_rank_final = exact_sink_rank_by_step(
    lfa_rows,
    sink_category=SINK_CATEGORY,
    final_step=FINAL_STEP,
)

source_step_sink_rates.to_csv(OUTPUT_DIR / 'lfa_to_hfa_exact_sink_rates_by_source_word_step.csv', index=False)
hfa_sink_rank_by_step_top.to_csv(OUTPUT_DIR / 'lfa_to_hfa_exact_sink_rank_by_step_top20.csv', index=False)
hfa_sink_rank_final.to_csv(OUTPUT_DIR / 'lfa_to_hfa_exact_sink_rank_final_step.csv', index=False)

hfa_sink_rank_final.head(50)

In [ ]:
thought_exact_flow = source_step_sink_rates[source_step_sink_rates['guess_norm'].eq(sink_norm)].copy()
thought_exact_by_source = (
    thought_exact_flow
    .groupby(['model_name', 'pipeline', 'original_word', 'word_norm'], observed=True)
    .agg(
        observed_steps_with_thought=('step', 'nunique'),
        max_step_thought_rate=('source_word_sink_rate', 'max'),
        total_thought_hits=('hit_count', 'sum'),
    )
    .reset_index()
    .sort_values(['pipeline', 'model_name', 'max_step_thought_rate'], ascending=[True, True, False])
)

thought_exact_flow.to_csv(OUTPUT_DIR / 'lfa_to_thought_exact_flow_by_source_word_step.csv', index=False)
thought_exact_by_source.to_csv(OUTPUT_DIR / 'lfa_to_thought_exact_flow_by_source_word.csv', index=False)
thought_exact_by_source.head(50)

# Option 3: Embedding Basin Sink Flow

Exact matching is strict. A chain can drift toward the `Thought` basin through guesses like `idea`, `belief`, `concept`, or `consciousness`. This maps each guess to its nearest high-frequency abstract target word using the existing embedding cache.

`NEAREST_HFA_THRESHOLD` is intentionally visible. Treat this as semantic-basin evidence, not literal exact-guess evidence.

In [ ]:
NEAREST_HFA_THRESHOLD = 0.45

nearest_hfa_map, missing_basin_embeddings = nearest_category_basin_map(
    guesses=sorted(lfa_rows['guess_norm'].dropna().astype(str).unique()),
    sink_vocab=high_freq_abstract_vocab,
    cache_path=EMBEDDING_CACHE,
)
nearest_hfa_map['passes_threshold'] = nearest_hfa_map['nearest_sink_similarity'].ge(NEAREST_HFA_THRESHOLD)
nearest_hfa_map.to_csv(OUTPUT_DIR / 'nearest_hfa_basin_map_for_lfa_guesses.csv', index=False)

print('Unique guesses:', len(nearest_hfa_map))
print('Missing embeddings:', len(missing_basin_embeddings))
nearest_hfa_map['nearest_sink_similarity'].describe()

In [ ]:
lfa_basin_rows, source_step_basin_rates, hfa_basin_rank_by_step_top, hfa_basin_rank_final = basin_rank_by_step(
    lfa_rows,
    basin_map=nearest_hfa_map,
    threshold=NEAREST_HFA_THRESHOLD,
    final_step=FINAL_STEP,
)

lfa_basin_rows['is_thought_embedding_basin'] = (
    lfa_basin_rows['nearest_sink_word'].eq(sink_norm)
    & lfa_basin_rows['is_embedding_basin']
)

source_step_basin_rates.to_csv(OUTPUT_DIR / 'lfa_to_hfa_embedding_basin_rates_by_source_word_step.csv', index=False)
hfa_basin_rank_by_step_top.to_csv(OUTPUT_DIR / 'lfa_to_hfa_embedding_basin_rank_by_step_top20.csv', index=False)
hfa_basin_rank_final.to_csv(OUTPUT_DIR / 'lfa_to_hfa_embedding_basin_rank_final_step.csv', index=False)

hfa_basin_rank_final.head(50)

In [ ]:
thought_basin_flow = source_step_basin_rates[source_step_basin_rates['nearest_sink_word'].eq(sink_norm)].copy()
thought_basin_by_source = (
    thought_basin_flow
    .groupby(['model_name', 'pipeline', 'original_word', 'word_norm'], observed=True)
    .agg(
        observed_steps_in_thought_basin=('step', 'nunique'),
        max_step_thought_basin_rate=('source_word_basin_rate', 'max'),
        mean_thought_basin_similarity=('mean_nearest_sink_similarity', 'mean'),
        total_thought_basin_hits=('hit_count', 'sum'),
    )
    .reset_index()
    .sort_values(['pipeline', 'model_name', 'max_step_thought_basin_rate'], ascending=[True, True, False])
)

thought_basin_flow.to_csv(OUTPUT_DIR / 'lfa_to_thought_embedding_basin_by_source_word_step.csv', index=False)
thought_basin_by_source.to_csv(OUTPUT_DIR / 'lfa_to_thought_embedding_basin_by_source_word.csv', index=False)
thought_basin_by_source.head(50)

# Option 4: Qualitative Chain Traces

Use these tables to write the qualitative story: does the chain generalize, simplify, become more common, or stabilize around a broader abstraction?

In [ ]:
example_trace = qualitative_trace(case_rows, named_sink_chain_summary)
if 'description' in example_trace.columns:
    display_cols = [
        'model_name', 'pipeline', 'original_word', 'instance_id', 'step',
        'guess_norm', 'guess_category_exact', 'exact_correct', 'is_named_sink_guess',
        'word_to_guess_similarity', 'description_short',
    ]
else:
    display_cols = example_trace.columns.tolist()

example_trace[display_cols].to_csv(OUTPUT_DIR / f'qualitative_trace_{source_norm}_example.csv', index=False)
example_trace[display_cols]

In [ ]:
transition_counts_top = guess_transitions(case_rows, top_n=20)
transition_counts_top.to_csv(OUTPUT_DIR / f'guess_transition_counts_{source_norm}_top20.csv', index=False)
transition_counts_top

# Framework for Addressing the First Question

Use three layers of evidence:

1. **Named sink evidence**: `named_sink_by_model` and `case_named_step_profile` answer whether `Solipsism` literally becomes `Thought`, by model/pipeline and by step.
2. **General attractor evidence**: `model_pipeline_hfa_flow` and `category_step_hfa_flow` show whether low-frequency abstract words generally move into high-frequency abstract vocabulary guesses.
3. **Semantic basin evidence**: `hfa_basin_rank_by_step_top`, `hfa_basin_rank_final`, and `thought_basin_by_source` show whether the drift moves into the neighborhood of high-frequency abstract words even without exact vocabulary matches.

For the qualitative paragraph, use `example_trace`, `case_top_guesses`, and `transition_counts_top` to show the actual path of decay instead of only saying that meaning decays.

# Second Question: How Does Meaning Decay?

A vague claim such as "meaning decays" can be avoided by describing the *type* of transformation. Four levels of analysis are used:

1. **Semantic loss**: exact accuracy, word-guess embedding similarity, and description similarity to step 0.
2. **Generalization / attractor movement**: guesses become more frequent, shorter, broader, or move into high-frequency abstract vocabulary/basins.
3. **Simplification / compression**: descriptions become shorter, less lexically diverse, more generic, or lose distinctive constraints.
4. **Neutralization / politeness**: descriptions lose affective or evaluative wording, or gain hedging and polite markers. A validated sentiment or politeness model is not included in this repository, so heuristic lexicon flags are used for qualitative inspection rather than as a primary statistical result.

Important aggregation rule: chain-level deltas are computed first, then source-word means are computed, and only then are category/model/pipeline means reported. This follows the same logic as `run_model_comparison_analysis.py`: avoid letting words with more rows or more volatile chains dominate category-level claims.


In [ ]:
frequency_lookup = load_frequency_lookup(PROJECT_ROOT)
frequency_lookup.to_csv(OUTPUT_DIR / 'frequency_lookup_for_qualitative_analysis.csv', index=False)
frequency_lookup.head()


## 2A. Source-Word Qualitative Profile

Start with the case-study word. This gives a concrete trace for prose: for `Solipsism`, does the chain become shorter, more common, more generic, or less semantically tied to the original description?


In [ ]:
case_qual_rows = add_description_text_features(case_rows)
case_qual_rows = add_guess_frequency_features(case_qual_rows, frequency_lookup)
case_qual_rows, missing_case_description_embeddings = add_description_similarity_to_step0(case_qual_rows, EMBEDDING_CACHE)

case_qual_rows.to_csv(OUTPUT_DIR / f'qualitative_metrics_by_step_{source_norm}.csv', index=False)
print('Missing description embeddings:', len(missing_case_description_embeddings))
case_qual_rows[[
    'model_name', 'pipeline', 'instance_id', 'step', 'guess_norm',
    'description_token_count', 'description_ttr', 'description_general_abstract_rate',
    'description_similarity_to_step0', 'guess_minus_source_zipf', 'word_to_guess_similarity',
]].head(11)


In [ ]:
QUAL_METRICS = [
    'description_char_len',
    'description_token_count',
    'description_ttr',
    'description_avg_token_len',
    'description_stopword_rate',
    'description_hedge_rate',
    'description_polite_rate',
    'description_affective_rate',
    'description_general_abstract_rate',
    'description_similarity_to_step0',
    'word_to_guess_similarity',
    'guess_minus_source_zipf',
    'guess_minus_source_lg10wf',
    'guess_minus_source_length',
]
QUAL_METRICS = [col for col in QUAL_METRICS if col in case_qual_rows.columns]

case_qual_deltas = chain_metric_deltas(case_qual_rows, QUAL_METRICS, baseline_step=0, final_step=FINAL_STEP)
case_delta_cols = [col for col in case_qual_deltas.columns if col.endswith(f'_delta_0_to_{FINAL_STEP}')]
case_qual_word_level, case_qual_by_model = aggregate_chain_deltas_word_first(case_qual_deltas, case_delta_cols)

case_qual_deltas.to_csv(OUTPUT_DIR / f'qualitative_chain_deltas_{source_norm}.csv', index=False)
case_qual_by_model.to_csv(OUTPUT_DIR / f'qualitative_deltas_by_model_{source_norm}.csv', index=False)
case_qual_by_model


In [ ]:
case_qual_step_profile = (
    case_qual_rows
    .groupby(['model_name', 'pipeline', 'step'], observed=True)
    .agg(
        chain_n=('instance_id', 'nunique'),
        exact_accuracy=('exact_correct', 'mean'),
        mean_word_guess_similarity=('word_to_guess_similarity', 'mean'),
        mean_description_similarity_to_step0=('description_similarity_to_step0', 'mean'),
        mean_description_tokens=('description_token_count', 'mean'),
        mean_description_ttr=('description_ttr', 'mean'),
        mean_general_abstract_rate=('description_general_abstract_rate', 'mean'),
        mean_affective_rate=('description_affective_rate', 'mean'),
        mean_hedge_rate=('description_hedge_rate', 'mean'),
        mean_polite_rate=('description_polite_rate', 'mean'),
        mean_guess_minus_source_zipf=('guess_minus_source_zipf', 'mean'),
        mean_guess_minus_source_length=('guess_minus_source_length', 'mean'),
        unique_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
    .sort_values(['pipeline', 'model_name', 'step'])
)

case_qual_step_profile.to_csv(OUTPUT_DIR / f'qualitative_step_profile_{source_norm}.csv', index=False)
case_qual_step_profile


## 2B. Generalization and Simplification Across Low-Frequency Abstract Words

This section uses all low-frequency abstract chains but starts with guess-side metrics, which are cheaper and still central to the qualitative claim. The key evidence is whether final guesses are more frequent/common than the source word, shorter, and less semantically similar to the source.


In [ ]:
lfa_guess_qual_rows, missing_lfa_guess_embeddings = add_word_guess_similarity(lfa_rows, EMBEDDING_CACHE)
lfa_guess_qual_rows = add_guess_frequency_features(lfa_guess_qual_rows, frequency_lookup)

LFA_GUESS_QUAL_METRICS = [
    'word_to_guess_similarity',
    'guess_minus_source_zipf',
    'guess_minus_source_lg10wf',
    'guess_minus_source_length',
]
LFA_GUESS_QUAL_METRICS = [col for col in LFA_GUESS_QUAL_METRICS if col in lfa_guess_qual_rows.columns]

lfa_guess_deltas = chain_metric_deltas(lfa_guess_qual_rows, LFA_GUESS_QUAL_METRICS, baseline_step=0, final_step=FINAL_STEP)
lfa_guess_delta_cols = [col for col in lfa_guess_deltas.columns if col.endswith(f'_delta_0_to_{FINAL_STEP}')]
lfa_guess_word_level, lfa_guess_by_category = aggregate_chain_deltas_word_first(lfa_guess_deltas, lfa_guess_delta_cols)

lfa_guess_qual_rows.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_metrics_by_step.csv', index=False)
lfa_guess_deltas.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_chain_deltas.csv', index=False)
lfa_guess_word_level.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_word_first_deltas.csv', index=False)
lfa_guess_by_category.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_category_deltas.csv', index=False)

print('Missing guess embeddings:', len(missing_lfa_guess_embeddings))
lfa_guess_by_category


In [ ]:
lfa_guess_step_word = (
    lfa_guess_qual_rows
    .groupby(['model_name', 'pipeline', 'category', 'original_word', 'word_norm', 'step'], observed=True)
    .agg(
        chain_n=('instance_id', 'nunique'),
        exact_accuracy=('exact_correct', 'mean'),
        mean_word_guess_similarity=('word_to_guess_similarity', 'mean'),
        mean_guess_minus_source_zipf=('guess_minus_source_zipf', 'mean'),
        mean_guess_minus_source_length=('guess_minus_source_length', 'mean'),
        unique_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
)

lfa_guess_step_category = (
    lfa_guess_step_word
    .groupby(['model_name', 'pipeline', 'category', 'step'], observed=True)
    .agg(
        source_word_n=('word_norm', 'nunique'),
        mean_exact_accuracy_across_words=('exact_accuracy', 'mean'),
        mean_word_guess_similarity_across_words=('mean_word_guess_similarity', 'mean'),
        mean_guess_minus_source_zipf_across_words=('mean_guess_minus_source_zipf', 'mean'),
        mean_guess_minus_source_length_across_words=('mean_guess_minus_source_length', 'mean'),
        mean_unique_guesses_across_words=('unique_guesses', 'mean'),
    )
    .reset_index()
    .sort_values(['pipeline', 'model_name', 'step'])
)

lfa_guess_step_word.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_word_step_profile.csv', index=False)
lfa_guess_step_category.to_csv(OUTPUT_DIR / 'lfa_guess_qualitative_category_step_profile.csv', index=False)
lfa_guess_step_category


## 2C. Description-Side Simplification Across Low-Frequency Abstract Words

This option is disabled by default because descriptions are loaded for every low-frequency abstract chain. It can be enabled when the full category-level description analysis is required.

This is the best place to support claims like: descriptions became shorter, lost lexical diversity, became more generic, or drifted away from the initial description.


In [ ]:
RUN_FULL_LFA_DESCRIPTION_QUAL = True#

if RUN_FULL_LFA_DESCRIPTION_QUAL:
    lfa_desc_rows = load_chain_rows(
        UNIFIED_CSV,
        word_to_category,
        source_categories=[SOURCE_CATEGORY],
        models=SELECTED_MODELS,
        pipelines=SELECTED_PIPELINES,
        include_descriptions=True,
        chunk_size=CHUNK_SIZE,
        progress=progress,
    )
    lfa_desc_rows = add_description_text_features(lfa_desc_rows)
    lfa_desc_rows, missing_lfa_desc_embeddings = add_description_similarity_to_step0(lfa_desc_rows, EMBEDDING_CACHE)

    LFA_DESC_METRICS = [
        'description_char_len',
        'description_token_count',
        'description_ttr',
        'description_avg_token_len',
        'description_stopword_rate',
        'description_hedge_rate',
        'description_polite_rate',
        'description_affective_rate',
        'description_general_abstract_rate',
        'description_similarity_to_step0',
    ]
    lfa_desc_deltas = chain_metric_deltas(lfa_desc_rows, LFA_DESC_METRICS, baseline_step=0, final_step=FINAL_STEP)
    lfa_desc_delta_cols = [col for col in lfa_desc_deltas.columns if col.endswith(f'_delta_0_to_{FINAL_STEP}')]
    lfa_desc_word_level, lfa_desc_by_category = aggregate_chain_deltas_word_first(lfa_desc_deltas, lfa_desc_delta_cols)

    lfa_desc_rows.to_csv(OUTPUT_DIR / 'lfa_description_qualitative_metrics_by_step.csv', index=False)
    lfa_desc_deltas.to_csv(OUTPUT_DIR / 'lfa_description_qualitative_chain_deltas.csv', index=False)
    lfa_desc_word_level.to_csv(OUTPUT_DIR / 'lfa_description_qualitative_word_first_deltas.csv', index=False)
    lfa_desc_by_category.to_csv(OUTPUT_DIR / 'lfa_description_qualitative_category_deltas.csv', index=False)

    display(lfa_desc_by_category)
else:
    print('Set RUN_FULL_LFA_DESCRIPTION_QUAL = True to load all LFA descriptions and run description-side category analysis.')


## 2D. Picking Qualitative Examples Instead of Cherry-Picking Blindly

A good qualitative section should say how examples were selected. These tables pick chains by measurable patterns: strongest move toward common words, largest semantic loss, and strongest simplification.


In [ ]:
# Source-word case-study examples.
case_example_rank = case_qual_deltas.copy()
for col in case_delta_cols:
    case_example_rank[col] = pd.to_numeric(case_example_rank[col], errors='coerce')

case_more_common_examples = case_example_rank.sort_values(
    f'guess_minus_source_zipf_delta_0_to_{FINAL_STEP}',
    ascending=False,
    na_position='last',
).head(20)

case_similarity_loss_examples = case_example_rank.sort_values(
    f'word_to_guess_similarity_delta_0_to_{FINAL_STEP}',
    ascending=True,
    na_position='last',
).head(20)

case_simplification_examples = case_example_rank.sort_values(
    f'description_token_count_delta_0_to_{FINAL_STEP}',
    ascending=True,
    na_position='last',
).head(20)

case_more_common_examples.to_csv(OUTPUT_DIR / f'example_selector_more_common_{source_norm}.csv', index=False)
case_similarity_loss_examples.to_csv(OUTPUT_DIR / f'example_selector_similarity_loss_{source_norm}.csv', index=False)
case_simplification_examples.to_csv(OUTPUT_DIR / f'example_selector_simplification_{source_norm}.csv', index=False)

case_more_common_examples.head()


In [ ]:
# Category-level examples selected after computing chain deltas, not by hand.
lfa_more_common_examples = lfa_guess_deltas.sort_values(
    f'guess_minus_source_zipf_delta_0_to_{FINAL_STEP}',
    ascending=False,
    na_position='last',
).head(50)

lfa_similarity_loss_examples = lfa_guess_deltas.sort_values(
    f'word_to_guess_similarity_delta_0_to_{FINAL_STEP}',
    ascending=True,
    na_position='last',
).head(50)

lfa_more_common_examples.to_csv(OUTPUT_DIR / 'lfa_example_selector_more_common_guesses.csv', index=False)
lfa_similarity_loss_examples.to_csv(OUTPUT_DIR / 'lfa_example_selector_similarity_loss.csv', index=False)

lfa_more_common_examples.head(20)


## Interpretations Supported by This Analysis

Possible answer templates after the outputs have been inspected:

- **Generalization**: "The final guesses were often semantically related but broader: low-frequency abstract targets moved toward common abstractions such as X/Y/Z. This is visible in the positive Zipf shift and the high-frequency-abstract sink/basin tables."
- **Simplification**: "Descriptions became shorter / less lexically diverse / more generic across steps, especially in pipeline __. The chain-level deltas show this before word-first averaging."
- **Attractor stabilization**: "Some chains did not simply degrade randomly; they converged on recurring guesses. The entropy/unique-guess and transition tables show repeated paths such as A -> B -> C."
- **Neutralization**: "Weak or heuristic evidence for neutralization is indicated when the affective rate decreases or the hedge rate increases. This should be treated as qualitative support unless a validated sentiment model or lexicon is added."

The strongest paper-safe structure is: report the category-level word-first table, then show two or three traces selected by the example-selector tables so the qualitative section is transparent rather than anecdotal.
